In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window

schema = "healthcare_project"

bronze_patients = spark.table(f"{schema}.bronze_patients")
bronze_encounters = spark.table(f"{schema}.bronze_encounters")
bronze_conditions = spark.table(f"{schema}.bronze_conditions")
bronze_procedures = spark.table(f"{schema}.bronze_procedures")
bronze_medications = spark.table(f"{schema}.bronze_medications")
bronze_providers = spark.table(f"{schema}.bronze_providers")

In [0]:
from pyspark.sql import functions as F
silver_patients = (
    bronze_patients
    .dropDuplicates(["Id"])
    .withColumnRenamed("Id", "patient_id")
    .withColumnRenamed("BIRTHDATE", "birth_date")
    .withColumnRenamed("DEATHDATE", "death_date")
    .withColumnRenamed("FIRST", "first_name")
    .withColumnRenamed("LAST", "last_name")
    .withColumnRenamed("GENDER", "gender")
    .withColumnRenamed("RACE", "race")
    .withColumnRenamed("ETHNICITY", "ethnicity")
    .withColumnRenamed("CITY", "city")
    .withColumnRenamed("STATE", "state")
    .withColumnRenamed("COUNTY", "county")
    .withColumn(
        "age",
        F.floor(F.datediff(F.current_date(), F.col("birth_date")) / 365.25)
    )
    .withColumn(
        "age_group",
        F.when(F.col("age") < 18, "Under 18")
         .when((F.col("age") >= 18) & (F.col("age") <= 35), "18-35")
         .when((F.col("age") >= 36) & (F.col("age") <= 50), "36-50")
         .when((F.col("age") >= 51) & (F.col("age") <= 65), "51-65")
         .otherwise("65+")
    )
    .withColumn("gender", F.upper(F.trim(F.col("gender"))))
    .withColumn("race", F.initcap(F.trim(F.col("race"))))
    .withColumn("ethnicity", F.initcap(F.trim(F.col("ethnicity"))))
    .withColumn("city", F.initcap(F.trim(F.col("city"))))
    .withColumn("state", F.upper(F.trim(F.col("state"))))
    .withColumn("county", F.initcap(F.trim(F.col("county"))))
)

silver_patients.write.mode("overwrite").saveAsTable(f"{schema}.silver_patients")
display(spark.table(f"{schema}.silver_patients").limit(5))

patient_id,birth_date,death_date,SSN,DRIVERS,PASSPORT,PREFIX,first_name,MIDDLE,last_name,SUFFIX,MAIDEN,MARITAL,race,ethnicity,gender,BIRTHPLACE,ADDRESS,city,state,county,FIPS,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,INCOME,ingestion_timestamp,data_source,age,age_group
4d87119a-8a1a-92a7-082d-a04b0de4c823,1965-09-23,2023-01-29,999-80-4959,S99976793,X65150149X,Mr.,Deangelo7,null,Hermann103,null,null,M,White,Hispanic,M,Stoneham Massachusetts US,808 Cummings Throughway,Boston,MASSACHUSETTS,Suffolk County,25025,2116,42.316570947818114,-71.07256128652682,240563.15,589820.19,173151,2026-04-06T23:16:41.215Z,synthea,60,51-65
4cca4bcb-30b0-9f8f-002c-c85b2fb2660d,2004-08-11,null,999-67-8302,S99914001,X28047206X,Mr.,Rosendo998,Clay913,Stracke611,null,null,null,White,Nonhispanic,M,Wilmington Massachusetts US,296 White View Apt 83,Andover,MASSACHUSETTS,Essex County,25009,1810,42.619042484536564,-71.19873497683236,49874.52,22801.32,37145,2026-04-06T23:16:41.215Z,synthea,21,18-35
390b8910-7168-76c0-a185-7b3423a355a7,2017-12-07,null,999-61-8018,null,null,null,Jason347,null,Carter549,null,null,null,White,Nonhispanic,M,Weymouth Massachusetts US,681 Turcotte Divide Unit 13,North Attleborough,MASSACHUSETTS,Bristol County,null,0,42.01588918862707,-71.3179296468958,23675.86,199.46,171616,2026-04-06T23:16:41.215Z,synthea,8,Under 18
55682631-7cd3-e5c5-419f-6971d1ab57ab,1986-04-05,null,999-55-5769,S99952417,X11454442X,Mrs.,Julio255,null,Metz686,null,DuBuque211,M,White,Nonhispanic,F,North Reading Massachusetts US,448 Wisoky Light Apt 58,Lowell,MASSACHUSETTS,Middlesex County,25017,1852,42.69321033334149,-71.30816937354338,14075.37,1092235.66,3335,2026-04-06T23:16:41.215Z,synthea,40,36-50
509a1afc-4a5f-d03a-ed4e-20021e81875e,1917-04-30,null,999-11-1882,S99974127,X65353518X,Mrs.,Shanon591,Martha161,Mraz590,null,Kilback373,M,White,Nonhispanic,F,Boston Massachusetts US,108 Baumbach Glen,Ashland,MASSACHUSETTS,Middlesex County,null,0,42.25940504021545,-71.49627657163029,477116.2,2064292.8,146322,2026-04-06T23:16:41.215Z,synthea,108,65+


In [0]:
silver_encounters = (
    bronze_encounters
    .dropDuplicates(["Id"])
    .withColumnRenamed("Id", "encounter_id")
    .withColumnRenamed("START", "encounter_start")
    .withColumnRenamed("STOP", "encounter_end")
    .withColumnRenamed("PATIENT", "patient_id")
    .withColumnRenamed("PROVIDER", "provider_id")
    .withColumnRenamed("PAYER", "payer_id")
    .withColumnRenamed("ENCOUNTERCLASS", "encounter_class")
    .withColumnRenamed("CODE", "encounter_code")
    .withColumnRenamed("DESCRIPTION", "encounter_description")
    .withColumnRenamed("BASE_ENCOUNTER_COST", "base_encounter_cost")
    .withColumnRenamed("TOTAL_CLAIM_COST", "total_claim_cost")
    .withColumnRenamed("PAYER_COVERAGE", "payer_coverage")
    .withColumn("encounter_date", F.to_date("encounter_start"))
    .withColumn("encounter_year", F.year("encounter_start"))
    .withColumn("encounter_month", F.month("encounter_start"))
    .withColumn(
        "length_of_stay_hours",
        (F.col("encounter_end").cast("long") - F.col("encounter_start").cast("long")) / 3600
    )
    .withColumn("encounter_class", F.lower(F.trim(F.col("encounter_class"))))
    .withColumn("encounter_description", F.initcap(F.trim(F.col("encounter_description"))))
    .fillna({"base_encounter_cost": 0.0, "total_claim_cost": 0.0, "payer_coverage": 0.0})
)

silver_encounters.write.mode("overwrite").saveAsTable(f"{schema}.silver_encounters")
display(spark.table(f"{schema}.silver_encounters").limit(5))

encounter_id,encounter_start,encounter_end,patient_id,ORGANIZATION,provider_id,payer_id,encounter_class,encounter_code,encounter_description,base_encounter_cost,total_claim_cost,payer_coverage,REASONCODE,REASONDESCRIPTION,ingestion_timestamp,data_source,encounter_date,encounter_year,encounter_month,length_of_stay_hours
ded304ea-11c4-b377-93cc-c68a057401a8,2025-11-05T00:10:04Z,2025-11-05T06:15:43Z,ded304ea-11c4-b377-906a-2fd8284f4370,48a6d3ab-50ef-349e-8945-2558a78f378e,648f7912-13d5-3613-af4e-772e6658beb1,df166300-5a78-3502-a46a-832842197811,ambulatory,185349003,Encounter For Check Up (procedure),85.55,4830.95,4780.95,103697008,Patient referral for dental care (procedure),2026-04-06T23:17:08.315Z,synthea,2025-11-05,2025,11,6.094166666666666
94945a34-7d9e-544d-7fc8-489889edd7b3,2025-10-19T20:11:07Z,2025-10-19T20:50:01Z,94945a34-7d9e-544d-bb42-76cbf0aafa90,817a1428-2045-3e89-9fa2-ef7d287d6fa5,53ac5de3-5cc3-3f54-83b6-d42a157be546,a735bf55-83e9-331a-899d-a82a60b9f60c,wellness,162673000,General Examination Of Patient (procedure),136.8,1063.94,851.14,null,null,2026-04-06T23:17:08.315Z,synthea,2025-10-19,2025,10,0.6483333333333333
562007be-38fa-27cb-5b54-de78e7a7e55c,2022-08-22T11:18:23Z,2022-08-22T11:33:23Z,562007be-38fa-27cb-120a-8d958137bde4,18061fd5-fda8-3e2d-87f0-0f3c87211bcc,97d2a9ff-8f9c-325b-a930-984da3ad09ae,df166300-5a78-3502-a46a-832842197811,ambulatory,424441002,Prenatal Initial Visit (regime/therapy),142.58,32839.87,32789.87,72892002,Normal pregnancy (finding),2026-04-06T23:17:08.315Z,synthea,2022-08-22,2022,8,0.25
1b9500cd-1086-91a7-db94-ac180b0af0e8,2018-05-29T04:28:11Z,2018-05-29T04:43:11Z,1b9500cd-1086-91a7-b2f9-d6224d8bf1c7,6170552c-4d27-3e64-8068-64a380f1837d,6d34bd22-1769-3610-9472-ebb0648c2c90,734afbd6-4794-363b-9bc0-6a3981533ed5,ambulatory,424619006,Prenatal Visit (regime/therapy),142.58,13673.03,13673.03,72892002,Normal pregnancy (finding),2026-04-06T23:17:08.315Z,synthea,2018-05-29,2018,5,0.25
751022eb-010a-f455-9db8-fe922d988535,2022-12-25T17:44:14Z,2022-12-25T17:59:14Z,751022eb-010a-f455-a71b-7ead7ae38ff9,a5577e5c-b886-37a0-8eee-8ff6feb7ea6b,4e03312c-dd79-3524-889a-ec8835a8aac5,b046940f-1664-3047-bca7-dfa76be352a4,ambulatory,185345009,Encounter For Symptom (procedure),85.55,85.55,0.0,444814009,Viral sinusitis (disorder),2026-04-06T23:17:08.315Z,synthea,2022-12-25,2022,12,0.25


In [0]:
spark.sql(f"SHOW TABLES IN {schema}").show(truncate=False)

+------------------+---------------------------------------+-----------+
|database          |tableName                              |isTemporary|
+------------------+---------------------------------------+-----------+
|healthcare_project|bronze_conditions                      |false      |
|healthcare_project|bronze_encounters                      |false      |
|healthcare_project|bronze_medications                     |false      |
|healthcare_project|bronze_patients                        |false      |
|healthcare_project|bronze_procedures                      |false      |
|healthcare_project|bronze_providers                       |false      |
|healthcare_project|gold_diagnosis_trends                  |false      |
|healthcare_project|gold_frequent_revisit_patients         |false      |
|healthcare_project|gold_medication_usage_by_age           |false      |
|healthcare_project|gold_patient_cost_journey              |false      |
|healthcare_project|gold_patient_encounter_summary 

In [0]:
from pyspark.sql import functions as F

schema = "healthcare_project"

bronze_conditions = spark.table(f"{schema}.bronze_conditions")
bronze_procedures = spark.table(f"{schema}.bronze_procedures")
bronze_medications = spark.table(f"{schema}.bronze_medications")
bronze_providers = spark.table(f"{schema}.bronze_providers")

In [0]:
silver_conditions = (
    bronze_conditions
    .dropDuplicates(["PATIENT", "ENCOUNTER", "CODE", "DESCRIPTION", "START"])
    .withColumnRenamed("START", "condition_start")
    .withColumnRenamed("STOP", "condition_end")
    .withColumnRenamed("PATIENT", "patient_id")
    .withColumnRenamed("ENCOUNTER", "encounter_id")
    .withColumnRenamed("SYSTEM", "condition_system")
    .withColumnRenamed("CODE", "condition_code")
    .withColumnRenamed("DESCRIPTION", "condition_description")
    .withColumn("condition_description", F.initcap(F.trim(F.col("condition_description"))))
)

silver_conditions.write.mode("overwrite").saveAsTable(f"{schema}.silver_conditions")
display(spark.table(f"{schema}.silver_conditions").limit(5))

condition_start,condition_end,patient_id,encounter_id,condition_system,condition_code,condition_description,ingestion_timestamp,data_source
2025-09-23,null,d4e8dd95-e0dc-ac29-3858-514a8031cc58,d4e8dd95-e0dc-ac29-25dc-6290b6327641,http://snomed.info/sct,160904001,Part-time Employment (finding),2026-04-06T23:17:29.722Z,synthea
2021-02-21,2021-03-10,5d80ca90-0981-e66b-8bd5-d876ec07a714,5d80ca90-0981-e66b-10ca-5649b4641c44,http://snomed.info/sct,384709000,Sprain (morphologic Abnormality),2026-04-06T23:17:29.722Z,synthea
2022-09-04,2022-09-04,f4fa81fe-97f6-63a0-f203-84953a4c3184,f4fa81fe-97f6-63a0-38fc-4b6a526030eb,http://snomed.info/sct,314529007,Medication Review Due (situation),2026-04-06T23:17:29.722Z,synthea
2025-02-20,2025-03-06,7aa4288e-ceb6-927d-0c01-d43f3ba40d00,7aa4288e-ceb6-927d-f9b7-4d42313d3073,http://snomed.info/sct,66383009,Gingivitis (disorder),2026-04-06T23:17:29.722Z,synthea
2016-06-29,2016-07-13,62359bbc-560a-657d-45df-f65a2f6633fe,62359bbc-560a-657d-2b24-67b3c0be5404,http://snomed.info/sct,195662009,Acute Viral Pharyngitis (disorder),2026-04-06T23:17:29.722Z,synthea


In [0]:
silver_procedures = (
    bronze_procedures
    .dropDuplicates(["PATIENT", "ENCOUNTER", "CODE", "DESCRIPTION", "START"])
    .withColumnRenamed("START", "procedure_start")
    .withColumnRenamed("STOP", "procedure_end")
    .withColumnRenamed("PATIENT", "patient_id")
    .withColumnRenamed("ENCOUNTER", "encounter_id")
    .withColumnRenamed("SYSTEM", "procedure_system")
    .withColumnRenamed("CODE", "procedure_code")
    .withColumnRenamed("DESCRIPTION", "procedure_description")
    .withColumnRenamed("BASE_COST", "base_cost")
    .withColumnRenamed("REASONCODE", "reason_code")
    .withColumnRenamed("REASONDESCRIPTION", "reason_description")
    .withColumn("procedure_description", F.initcap(F.trim(F.col("procedure_description"))))
    .withColumn("reason_description", F.initcap(F.trim(F.col("reason_description"))))
    .fillna({"base_cost": 0.0})
)

silver_procedures.write.mode("overwrite").saveAsTable(f"{schema}.silver_procedures")
display(spark.table(f"{schema}.silver_procedures").limit(5))

procedure_start,procedure_end,patient_id,encounter_id,procedure_system,procedure_code,procedure_description,base_cost,reason_code,reason_description,ingestion_timestamp,data_source
2021-12-13T09:12:28Z,2021-12-13T11:38:28Z,f9e12a9c-5eaf-2826-49d3-8feaa111b6e2,f9e12a9c-5eaf-2826-ccad-7ea77ca4f909,http://snomed.info/sct,265764009,Renal Dialysis (procedure),1705.48,431857002,Chronic Kidney Disease Stage 4 (disorder),2026-04-06T23:17:58.905Z,synthea
2024-02-24T13:25:28Z,2024-02-24T14:18:20Z,f9e12a9c-5eaf-2826-49d3-8feaa111b6e2,f9e12a9c-5eaf-2826-715e-52203eb834d5,http://snomed.info/sct,710824005,Assessment Of Health And Social Care Needs (procedure),431.4,null,null,2026-04-06T23:17:58.905Z,synthea
2017-04-05T20:30:05Z,2017-04-05T20:42:11Z,4a146db7-5171-d950-17f0-61e93713f7b5,4a146db7-5171-d950-a78d-95667b3ffbc4,http://snomed.info/sct,171207006,Depression Screening (procedure),431.4,null,null,2026-04-06T23:17:58.905Z,synthea
2025-11-26T20:01:51Z,2025-11-26T20:16:51Z,4a146db7-5171-d950-17f0-61e93713f7b5,4a146db7-5171-d950-3203-ccccb2e49d08,http://snomed.info/sct,28163009,Skin Test For Tuberculosis Tine Test (procedure),3451.2,72892002,Normal Pregnancy (finding),2026-04-06T23:17:58.905Z,synthea
2025-12-24T20:01:51Z,2025-12-24T20:16:51Z,4a146db7-5171-d950-17f0-61e93713f7b5,4a146db7-5171-d950-7821-fbb891cb0f98,http://snomed.info/sct,443529005,Detection Of Chromosomal Aneuploidy In Prenatal Amniotic Fluid Specimen Using Fluorescence In Situ Hybridization Screening Technique (procedure),862.8,72892002,Normal Pregnancy (finding),2026-04-06T23:17:58.905Z,synthea


In [0]:
silver_medications = (
    bronze_medications
    .dropDuplicates(["PATIENT", "ENCOUNTER", "CODE", "DESCRIPTION", "START"])
    .withColumnRenamed("START", "medication_start")
    .withColumnRenamed("STOP", "medication_end")
    .withColumnRenamed("PATIENT", "patient_id")
    .withColumnRenamed("PAYER", "payer_id")
    .withColumnRenamed("ENCOUNTER", "encounter_id")
    .withColumnRenamed("CODE", "medication_code")
    .withColumnRenamed("DESCRIPTION", "medication_description")
    .withColumnRenamed("BASE_COST", "base_cost")
    .withColumnRenamed("PAYER_COVERAGE", "payer_coverage")
    .withColumnRenamed("DISPENSES", "dispenses")
    .withColumnRenamed("TOTALCOST", "total_cost")
    .withColumnRenamed("REASONCODE", "reason_code")
    .withColumnRenamed("REASONDESCRIPTION", "reason_description")
    .withColumn("medication_description", F.initcap(F.trim(F.col("medication_description"))))
    .withColumn("reason_description", F.initcap(F.trim(F.col("reason_description"))))
    .fillna({"base_cost": 0.0, "payer_coverage": 0.0, "total_cost": 0.0, "dispenses": 0})
)

silver_medications.write.mode("overwrite").saveAsTable(f"{schema}.silver_medications")
display(spark.table(f"{schema}.silver_medications").limit(5))

medication_start,medication_end,patient_id,payer_id,encounter_id,medication_code,medication_description,base_cost,payer_coverage,dispenses,total_cost,reason_code,reason_description,ingestion_timestamp,data_source
2016-06-04T14:36:43Z,2017-06-10T14:36:43Z,862f86b7-3069-c259-5f22-b2876f96aa28,d31fccc3-1767-390d-966a-22a5156f4219,862f86b7-3069-c259-6fce-52db20594b35,744842,Raltegravir 400 Mg Oral Tablet,129.94,103.95,12,1559.28,86406008,Human Immunodeficiency Virus Infection (disorder),2026-04-06T23:18:33.056Z,synthea
2024-07-22T14:36:43Z,2025-08-02T14:36:43Z,862f86b7-3069-c259-5f22-b2876f96aa28,d31fccc3-1767-390d-966a-22a5156f4219,862f86b7-3069-c259-10cb-6afa1e9fe049,314231,Simvastatin 10 Mg Oral Tablet,0.43,0.34,12,5.16,55822004,Hyperlipidemia (disorder),2026-04-06T23:18:33.056Z,synthea
2017-11-20T04:18:21Z,2018-11-26T04:18:21Z,e1b9b0ae-1df0-8258-5e61-2bf855a47755,a735bf55-83e9-331a-899d-a82a60b9f60c,e1b9b0ae-1df0-8258-ddb4-85b3a461fd8a,314076,Lisinopril 10 Mg Oral Tablet,0.91,0.0,4,3.64,59621000,Essential Hypertension (disorder),2026-04-06T23:18:33.056Z,synthea
2020-03-25T00:19:42Z,2020-03-25T00:19:42Z,57637642-886c-c38b-e37e-321df139d184,26aab0cd-6aba-3e1b-ac5b-05c8867e762c,57637642-886c-c38b-6463-e5b4f9179588,205923,1 Ml Epoetin Alfa 4000 Unt/ml Injection [epogen],29.21,29.21,1,29.21,271737000,Anemia (disorder),2026-04-06T23:18:33.056Z,synthea
2020-04-06T10:29:42Z,2020-04-06T10:29:42Z,57637642-886c-c38b-e37e-321df139d184,26aab0cd-6aba-3e1b-ac5b-05c8867e762c,57637642-886c-c38b-1bca-8b1cf201c57a,205923,1 Ml Epoetin Alfa 4000 Unt/ml Injection [epogen],29.21,29.21,1,29.21,271737000,Anemia (disorder),2026-04-06T23:18:33.056Z,synthea


In [0]:
silver_providers = (
    bronze_providers
    .dropDuplicates(["Id"])
    .withColumnRenamed("Id", "provider_id")
    .withColumnRenamed("ORGANIZATION", "organization_id")
    .withColumnRenamed("NAME", "provider_name")
    .withColumnRenamed("GENDER", "gender")
    .withColumnRenamed("SPECIALITY", "speciality")
    .withColumnRenamed("ADDRESS", "address")
    .withColumnRenamed("CITY", "city")
    .withColumnRenamed("STATE", "state")
    .withColumnRenamed("ZIP", "zip")
    .withColumnRenamed("LAT", "lat")
    .withColumnRenamed("LON", "lon")
    .withColumnRenamed("ENCOUNTERS", "encounter_count")
    .withColumnRenamed("PROCEDURES", "procedure_count")
    .withColumn("provider_name", F.initcap(F.trim(F.col("provider_name"))))
    .withColumn("gender", F.upper(F.trim(F.col("gender"))))
    .withColumn("speciality", F.initcap(F.trim(F.col("speciality"))))
    .withColumn("address", F.initcap(F.trim(F.col("address"))))
    .withColumn("city", F.initcap(F.trim(F.col("city"))))
    .withColumn("state", F.upper(F.trim(F.col("state"))))
)

silver_providers.write.mode("overwrite").saveAsTable(f"{schema}.silver_providers")
display(spark.table(f"{schema}.silver_providers").limit(5))

provider_id,organization_id,provider_name,gender,speciality,address,city,state,zip,lat,lon,encounter_count,procedure_count,ingestion_timestamp,data_source
12b04aba-c7fb-310f-8a30-286c5f64d59f,6b2e5b97-70f6-3219-8c99-86de138e6808,Forest317 Rutherford999,M,General Practice,123 Summer St,Worcester,MA,1608,42.26496835,-71.79655403681465,404,0,2026-04-06T23:18:48.104Z,synthea
33a98b93-fb72-36e0-96e6-9072f8c12802,37cba62c-ea42-3930-a1e1-7a285ace4b8a,Nolan344 Flatley871,M,General Practice,58 Princeton Rd,Malden,MA,21484411,42.4367715,-71.05428049538298,34,0,2026-04-06T23:18:48.104Z,synthea
ed78f1e3-e91f-3d05-bd52-365659537144,49fa72da-905b-3258-b22a-06e007c74c81,Hung902 Weissnat378,M,General Practice,30 Capital Dr Ste A,Westfield,MA,10851832,42.1390327,-72.7584859,102,0,2026-04-06T23:18:48.104Z,synthea
52290c11-50e3-3854-80a0-6cba40a15359,719db77c-034f-3079-9b12-0d2ec2efeef9,Staci390 Murray856,F,General Practice,1601 Washington St,Boston,MA,21182006,42.3381394,-71.0748562,254,0,2026-04-06T23:18:48.104Z,synthea
8dc6d874-e7de-36f3-adbd-438ebae41022,05cc5056-7af4-3afb-9a5a-d7a1c23a2509,Francis500 Lehner980,M,General Practice,2069 Roosevelt Avenue,West Springfield,MA,10892595,42.105237,-72.6213712,155,0,2026-04-06T23:18:48.104Z,synthea


In [0]:
silver_tables = [
    "silver_patients",
    "silver_encounters",
    "silver_conditions",
    "silver_procedures",
    "silver_medications",
    "silver_providers"
]

for table in silver_tables:
    print(f"\nPreview: {table}")
    display(spark.table(f"healthcare_project.{table}").limit(5))


Preview: silver_patients


patient_id,birth_date,death_date,SSN,DRIVERS,PASSPORT,PREFIX,first_name,MIDDLE,last_name,SUFFIX,MAIDEN,MARITAL,race,ethnicity,gender,BIRTHPLACE,ADDRESS,city,state,county,FIPS,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,INCOME,ingestion_timestamp,data_source,age,age_group
4d87119a-8a1a-92a7-082d-a04b0de4c823,1965-09-23,2023-01-29,999-80-4959,S99976793,X65150149X,Mr.,Deangelo7,null,Hermann103,null,null,M,White,Hispanic,M,Stoneham Massachusetts US,808 Cummings Throughway,Boston,MASSACHUSETTS,Suffolk County,25025,2116,42.316570947818114,-71.07256128652682,240563.15,589820.19,173151,2026-04-06T23:16:41.215Z,synthea,60,51-65
4cca4bcb-30b0-9f8f-002c-c85b2fb2660d,2004-08-11,null,999-67-8302,S99914001,X28047206X,Mr.,Rosendo998,Clay913,Stracke611,null,null,null,White,Nonhispanic,M,Wilmington Massachusetts US,296 White View Apt 83,Andover,MASSACHUSETTS,Essex County,25009,1810,42.619042484536564,-71.19873497683236,49874.52,22801.32,37145,2026-04-06T23:16:41.215Z,synthea,21,18-35
390b8910-7168-76c0-a185-7b3423a355a7,2017-12-07,null,999-61-8018,null,null,null,Jason347,null,Carter549,null,null,null,White,Nonhispanic,M,Weymouth Massachusetts US,681 Turcotte Divide Unit 13,North Attleborough,MASSACHUSETTS,Bristol County,null,0,42.01588918862707,-71.3179296468958,23675.86,199.46,171616,2026-04-06T23:16:41.215Z,synthea,8,Under 18
55682631-7cd3-e5c5-419f-6971d1ab57ab,1986-04-05,null,999-55-5769,S99952417,X11454442X,Mrs.,Julio255,null,Metz686,null,DuBuque211,M,White,Nonhispanic,F,North Reading Massachusetts US,448 Wisoky Light Apt 58,Lowell,MASSACHUSETTS,Middlesex County,25017,1852,42.69321033334149,-71.30816937354338,14075.37,1092235.66,3335,2026-04-06T23:16:41.215Z,synthea,40,36-50
509a1afc-4a5f-d03a-ed4e-20021e81875e,1917-04-30,null,999-11-1882,S99974127,X65353518X,Mrs.,Shanon591,Martha161,Mraz590,null,Kilback373,M,White,Nonhispanic,F,Boston Massachusetts US,108 Baumbach Glen,Ashland,MASSACHUSETTS,Middlesex County,null,0,42.25940504021545,-71.49627657163029,477116.2,2064292.8,146322,2026-04-06T23:16:41.215Z,synthea,108,65+



Preview: silver_encounters


encounter_id,encounter_start,encounter_end,patient_id,ORGANIZATION,provider_id,payer_id,encounter_class,encounter_code,encounter_description,base_encounter_cost,total_claim_cost,payer_coverage,REASONCODE,REASONDESCRIPTION,ingestion_timestamp,data_source,encounter_date,encounter_year,encounter_month,length_of_stay_hours
ded304ea-11c4-b377-93cc-c68a057401a8,2025-11-05T00:10:04Z,2025-11-05T06:15:43Z,ded304ea-11c4-b377-906a-2fd8284f4370,48a6d3ab-50ef-349e-8945-2558a78f378e,648f7912-13d5-3613-af4e-772e6658beb1,df166300-5a78-3502-a46a-832842197811,ambulatory,185349003,Encounter For Check Up (procedure),85.55,4830.95,4780.95,103697008,Patient referral for dental care (procedure),2026-04-06T23:17:08.315Z,synthea,2025-11-05,2025,11,6.094166666666666
94945a34-7d9e-544d-7fc8-489889edd7b3,2025-10-19T20:11:07Z,2025-10-19T20:50:01Z,94945a34-7d9e-544d-bb42-76cbf0aafa90,817a1428-2045-3e89-9fa2-ef7d287d6fa5,53ac5de3-5cc3-3f54-83b6-d42a157be546,a735bf55-83e9-331a-899d-a82a60b9f60c,wellness,162673000,General Examination Of Patient (procedure),136.8,1063.94,851.14,null,null,2026-04-06T23:17:08.315Z,synthea,2025-10-19,2025,10,0.6483333333333333
562007be-38fa-27cb-5b54-de78e7a7e55c,2022-08-22T11:18:23Z,2022-08-22T11:33:23Z,562007be-38fa-27cb-120a-8d958137bde4,18061fd5-fda8-3e2d-87f0-0f3c87211bcc,97d2a9ff-8f9c-325b-a930-984da3ad09ae,df166300-5a78-3502-a46a-832842197811,ambulatory,424441002,Prenatal Initial Visit (regime/therapy),142.58,32839.87,32789.87,72892002,Normal pregnancy (finding),2026-04-06T23:17:08.315Z,synthea,2022-08-22,2022,8,0.25
1b9500cd-1086-91a7-db94-ac180b0af0e8,2018-05-29T04:28:11Z,2018-05-29T04:43:11Z,1b9500cd-1086-91a7-b2f9-d6224d8bf1c7,6170552c-4d27-3e64-8068-64a380f1837d,6d34bd22-1769-3610-9472-ebb0648c2c90,734afbd6-4794-363b-9bc0-6a3981533ed5,ambulatory,424619006,Prenatal Visit (regime/therapy),142.58,13673.03,13673.03,72892002,Normal pregnancy (finding),2026-04-06T23:17:08.315Z,synthea,2018-05-29,2018,5,0.25
751022eb-010a-f455-9db8-fe922d988535,2022-12-25T17:44:14Z,2022-12-25T17:59:14Z,751022eb-010a-f455-a71b-7ead7ae38ff9,a5577e5c-b886-37a0-8eee-8ff6feb7ea6b,4e03312c-dd79-3524-889a-ec8835a8aac5,b046940f-1664-3047-bca7-dfa76be352a4,ambulatory,185345009,Encounter For Symptom (procedure),85.55,85.55,0.0,444814009,Viral sinusitis (disorder),2026-04-06T23:17:08.315Z,synthea,2022-12-25,2022,12,0.25



Preview: silver_conditions


...

procedure_start,procedure_end,patient_id,encounter_id,procedure_system,procedure_code,procedure_description,base_cost,reason_code,reason_description,ingestion_timestamp,data_source
2021-12-13T09:12:28Z,2021-12-13T11:38:28Z,f9e12a9c-5eaf-2826-49d3-8feaa111b6e2,f9e12a9c-5eaf-2826-ccad-7ea77ca4f909,http://snomed.info/sct,265764009,Renal Dialysis (procedure),1705.48,431857002,Chronic Kidney Disease Stage 4 (disorder),2026-04-06T23:17:58.905Z,synthea
2024-02-24T13:25:28Z,2024-02-24T14:18:20Z,f9e12a9c-5eaf-2826-49d3-8feaa111b6e2,f9e12a9c-5eaf-2826-715e-52203eb834d5,http://snomed.info/sct,710824005,Assessment Of Health And Social Care Needs (procedure),431.4,null,null,2026-04-06T23:17:58.905Z,synthea
2017-04-05T20:30:05Z,2017-04-05T20:42:11Z,4a146db7-5171-d950-17f0-61e93713f7b5,4a146db7-5171-d950-a78d-95667b3ffbc4,http://snomed.info/sct,171207006,Depression Screening (procedure),431.4,null,null,2026-04-06T23:17:58.905Z,synthea
2025-11-26T20:01:51Z,2025-11-26T20:16:51Z,4a146db7-5171-d950-17f0-61e93713f7b5,4a146db7-5171-d950-3203-ccccb2e49d08,http://snomed.info/sct,28163009,Skin Test For Tuberculosis Tine Test (procedure),3451.2,72892002,Normal Pregnancy (finding),2026-04-06T23:17:58.905Z,synthea
2025-12-24T20:01:51Z,2025-12-24T20:16:51Z,4a146db7-5171-d950-17f0-61e93713f7b5,4a146db7-5171-d950-7821-fbb891cb0f98,http://snomed.info/sct,443529005,Detection Of Chromosomal Aneuploidy In Prenatal Amniotic Fluid Specimen Using Fluorescence In Situ Hybridization Screening Technique (procedure),862.8,72892002,Normal Pregnancy (finding),2026-04-06T23:17:58.905Z,synthea



Preview: silver_medications


medication_start,medication_end,patient_id,payer_id,encounter_id,medication_code,medication_description,base_cost,payer_coverage,dispenses,total_cost,reason_code,reason_description,ingestion_timestamp,data_source
2016-06-04T14:36:43Z,2017-06-10T14:36:43Z,862f86b7-3069-c259-5f22-b2876f96aa28,d31fccc3-1767-390d-966a-22a5156f4219,862f86b7-3069-c259-6fce-52db20594b35,744842,Raltegravir 400 Mg Oral Tablet,129.94,103.95,12,1559.28,86406008,Human Immunodeficiency Virus Infection (disorder),2026-04-06T23:18:33.056Z,synthea
2024-07-22T14:36:43Z,2025-08-02T14:36:43Z,862f86b7-3069-c259-5f22-b2876f96aa28,d31fccc3-1767-390d-966a-22a5156f4219,862f86b7-3069-c259-10cb-6afa1e9fe049,314231,Simvastatin 10 Mg Oral Tablet,0.43,0.34,12,5.16,55822004,Hyperlipidemia (disorder),2026-04-06T23:18:33.056Z,synthea
2017-11-20T04:18:21Z,2018-11-26T04:18:21Z,e1b9b0ae-1df0-8258-5e61-2bf855a47755,a735bf55-83e9-331a-899d-a82a60b9f60c,e1b9b0ae-1df0-8258-ddb4-85b3a461fd8a,314076,Lisinopril 10 Mg Oral Tablet,0.91,0.0,4,3.64,59621000,Essential Hypertension (disorder),2026-04-06T23:18:33.056Z,synthea
2020-03-25T00:19:42Z,2020-03-25T00:19:42Z,57637642-886c-c38b-e37e-321df139d184,26aab0cd-6aba-3e1b-ac5b-05c8867e762c,57637642-886c-c38b-6463-e5b4f9179588,205923,1 Ml Epoetin Alfa 4000 Unt/ml Injection [epogen],29.21,29.21,1,29.21,271737000,Anemia (disorder),2026-04-06T23:18:33.056Z,synthea
2020-04-06T10:29:42Z,2020-04-06T10:29:42Z,57637642-886c-c38b-e37e-321df139d184,26aab0cd-6aba-3e1b-ac5b-05c8867e762c,57637642-886c-c38b-1bca-8b1cf201c57a,205923,1 Ml Epoetin Alfa 4000 Unt/ml Injection [epogen],29.21,29.21,1,29.21,271737000,Anemia (disorder),2026-04-06T23:18:33.056Z,synthea



Preview: silver_providers


provider_id,organization_id,provider_name,gender,speciality,address,city,state,zip,lat,lon,encounter_count,procedure_count,ingestion_timestamp,data_source
12b04aba-c7fb-310f-8a30-286c5f64d59f,6b2e5b97-70f6-3219-8c99-86de138e6808,Forest317 Rutherford999,M,General Practice,123 Summer St,Worcester,MA,1608,42.26496835,-71.79655403681465,404,0,2026-04-06T23:18:48.104Z,synthea
33a98b93-fb72-36e0-96e6-9072f8c12802,37cba62c-ea42-3930-a1e1-7a285ace4b8a,Nolan344 Flatley871,M,General Practice,58 Princeton Rd,Malden,MA,21484411,42.4367715,-71.05428049538298,34,0,2026-04-06T23:18:48.104Z,synthea
ed78f1e3-e91f-3d05-bd52-365659537144,49fa72da-905b-3258-b22a-06e007c74c81,Hung902 Weissnat378,M,General Practice,30 Capital Dr Ste A,Westfield,MA,10851832,42.1390327,-72.7584859,102,0,2026-04-06T23:18:48.104Z,synthea
52290c11-50e3-3854-80a0-6cba40a15359,719db77c-034f-3079-9b12-0d2ec2efeef9,Staci390 Murray856,F,General Practice,1601 Washington St,Boston,MA,21182006,42.3381394,-71.0748562,254,0,2026-04-06T23:18:48.104Z,synthea
8dc6d874-e7de-36f3-adbd-438ebae41022,05cc5056-7af4-3afb-9a5a-d7a1c23a2509,Francis500 Lehner980,M,General Practice,2069 Roosevelt Avenue,West Springfield,MA,10892595,42.105237,-72.6213712,155,0,2026-04-06T23:18:48.104Z,synthea


In [0]:
spark.sql(f"SHOW TABLES IN {schema}").show(truncate=False)

+------------------+---------------------------------------+-----------+
|database          |tableName                              |isTemporary|
+------------------+---------------------------------------+-----------+
|healthcare_project|bronze_conditions                      |false      |
|healthcare_project|bronze_encounters                      |false      |
|healthcare_project|bronze_medications                     |false      |
|healthcare_project|bronze_patients                        |false      |
|healthcare_project|bronze_procedures                      |false      |
|healthcare_project|bronze_providers                       |false      |
|healthcare_project|gold_diagnosis_trends                  |false      |
|healthcare_project|gold_frequent_revisit_patients         |false      |
|healthcare_project|gold_medication_usage_by_age           |false      |
|healthcare_project|gold_patient_cost_journey              |false      |
|healthcare_project|gold_patient_encounter_summary 